# Medallion Pipeline - Ad Events Processing

## Objective

Build a Medallion Architecture ETL pipeline to ingest, clean, validate and transform vendor event data using Databricks and Delta Lake.

The pipeline follows Bronze, Silver and Gold layers to improve data quality, scalability and reporting readiness.

# Step 1: Data Ingestion

## Bronze Layer - Raw Data Loading

In this step, the source CSV files are loaded into Spark DataFrames.

Files processed:

- advertisers.csv
- events_day1.csv
- events_day2.csv

## Approach

- Read CSV files from the Databricks Volume location.
- Enable header extraction.
- Enable schema inference for initial exploration.
- Review schema, sample records and record counts before transformation.

## Purpose

This step helps understand:
- Source data structure
- Column names
- Data types
- Record volume
- Schema differences between event files

In [0]:
# Read advertisers.csv
advertisers_df = spark.read.csv(
    "/Volumes/workspace/default/assessment_files/Datalytics/advertisers.csv",
    header=True,
    inferSchema=True
)

# Read events_day1.csv
events_day1_df = spark.read.csv(
    "/Volumes/workspace/default/assessment_files/Datalytics/events_day1.csv",
    header=True,
    inferSchema=True
)

# Read events_day2.csv
events_day2_df = spark.read.csv(
    "/Volumes/workspace/default/assessment_files/Datalytics/events_day2.csv",
    header=True,
    inferSchema=True
)

print("Advertisers Schema")
advertisers_df.printSchema()
advertisers_df.show(5)
print("Advertisers record count:",advertisers_df.count())

print("Events Day1 Schema")
events_day1_df.printSchema()
events_day1_df.show(5)
print("Events day 1 record count:",events_day1_df.count())

print("Events Day2 Schema")
events_day2_df.printSchema()
events_day2_df.show(5)
print("Events day 2 record count:",events_day2_df.count())

#Display column names
print("\nAdvertisers columns:",advertisers_df.columns)
print("\nEvents Day1 columns:", events_day1_df.columns)
print("\nEvents Day2 columns:", events_day2_df.columns)

Advertisers Schema
root
 |-- advertiser_id: string (nullable = true)
 |-- advertiser_name: string (nullable = true)
 |-- vertical: string (nullable = true)
 |-- region: string (nullable = true)
 |-- account_manager: string (nullable = true)

+-------------+---------------+----------+------+---------------+
|advertiser_id|advertiser_name|  vertical|region|account_manager|
+-------------+---------------+----------+------+---------------+
|     adv_1265|      PrimeLabs|    Retail|  APAC|         marcus|
|     adv_1272|      BlueWorks|     Media|    NA|            tom|
|     adv_1279|      RidgeLabs|Healthcare|    NA|           NULL|
|     adv_1293|    NorthRetail|Healthcare|    NA|            tom|
|     adv_1300|       NorthPay|    Travel|  EMEA|           lena|
+-------------+---------------+----------+------+---------------+
only showing top 5 rows
Advertisers record count: 10
Events Day1 Schema
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- 

# Data Profiling Observations

After loading the source files, the following checks were performed:

- Reviewed schemas for all input datasets.
- Compared event_day1 and event_day2 structures.
- Identified potential schema differences.
- Verified record counts before transformation.

The observations from this step are used for schema alignment in the Silver layer.

# Step 2: Bronze Layer

### Objective
Ingest the raw CSV files into Delta tables without applying any transformations.

### Why?
The Bronze layer preserves the original source data (raw fidelity) and serves as the source of truth for downstream Silver and Gold processing.

### Tables Created

- advertisers_bronze
- events_day1_bronze
- events_day2_bronze

In [0]:

# To check what schemas/databases are available.
spark.catalog.listDatabases() 

# Save raw data in Bronze zone 

advertisers_df.write.format("delta").mode("overwrite").saveAsTable("advertisers_bronze")
events_day1_df.write.format("delta").mode("overwrite").saveAsTable("events_day1_bronze")
events_day2_df.write.format("delta").mode("overwrite").saveAsTable("events_day2_bronze")




















# Bronze Layer Verification

### Objective
Read the Bronze tables after ingestion to verify that the raw data has been successfully stored as Delta tables without applying any transformations.

### Validation Checks
- Confirm Delta tables are created successfully.
- Verify sample records.
- Confirm schema is preserved.

In [0]:

# List all Bronze tables
spark.sql("SHOW TABLES").show()

print("\nAdvertisers Bronze")
spark.read.table("advertisers_bronze").show(5)

print("\nEvents Day1 Bronze")
spark.read.table("events_day1_bronze").show(5)

print("\nEvents Day2 Bronze")
spark.read.table("events_day2_bronze").show(5)

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|  advertisers_bronze|      false|
| default|  events_day1_bronze|      false|
| default|  events_day2_bronze|      false|
| default|   events_quarantine|      false|
| default|       events_silver|      false|
| default|gold_daily_advert...|      false|
+--------+--------------------+-----------+


Advertisers Bronze
+-------------+---------------+----------+------+---------------+
|advertiser_id|advertiser_name|  vertical|region|account_manager|
+-------------+---------------+----------+------+---------------+
|     adv_1265|      PrimeLabs|    Retail|  APAC|         marcus|
|     adv_1272|      BlueWorks|     Media|    NA|            tom|
|     adv_1279|      RidgeLabs|Healthcare|    NA|           NULL|
|     adv_1293|    NorthRetail|Healthcare|    NA|            tom|
|     adv_1300|       NorthPay|    Travel|  EMEA|           lena|
+----------

# Step 3: Silver Layer

### Objective

Read the Bronze Delta tables and create a single clean events table.

The Silver layer is responsible for handling schema drift, data quality issues, deduplication and timestamp standardization while keeping the Bronze layer unchanged.

### Transformations Performed

- Standardize column names between event versions.
- Align schemas before combining datasets.
- Cast columns into correct data types.
- Remove duplicate events.
- Validate mandatory fields.
- Separate invalid records.

## Data Quality Validation and Quarantine

Before creating the Silver table, incoming records are validated.

The following rules are applied:

### Timestamp Standardization

- Source files may contain timestamps with different timezone formats.
- All timestamps are normalized to UTC to ensure consistent ordering and comparison.

### Quarantine Rules

Records are moved to a separate quarantine table instead of being dropped.

Records are quarantined when:

1. ingest_time is malformed or cannot be converted to a valid timestamp.
2. event_time is malformed (for example, "N/A").
3. media_cost contains negative values.

This allows investigation and correction of bad source data while preserving the original records.

In [0]:
from pyspark.sql import functions as F

# Set Spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Normalize ingest_time to UTC
combined_df = combined_df.withColumn(
    "ingest_time_utc",
    F.to_utc_timestamp(
        F.to_timestamp("ingest_time"),
        "Asia/Kolkata"
    )
)

# Data quality checks
bad_ingest_time = F.col("ingest_time_utc").isNull()

# Validate event_time.
# Using try_to_timestamp() instead of to_timestamp() because malformed
# values (e.g., "N/A") would cause CAST_INVALID_INPUT errors.
# try_to_timestamp() safely returns NULL for invalid timestamps,
# allowing such records to be quarantined.

bad_event_time = (
    (F.col("event_time") == "N/A") |
    F.expr("try_to_timestamp(event_time)").isNull()  
)

bad_cost = F.col("media_cost") < 0

# Quarantine invalid records
quarantine_df = combined_df.filter(
    bad_ingest_time |
    bad_event_time |
    bad_cost
)
#used append instead of overwrite since quarantine usually  preserve history.
quarantine_df.write.format("delta").mode("append").saveAsTable("events_quarantine")

# Keep only valid records
clean_df = combined_df.filter(
    ~(bad_ingest_time | bad_event_time | bad_cost)
)




## Deduplication

The Silver layer handles two types of duplicates:

### 1. Exact Duplicate Records

Identical rows generated due to repeated ingestion are removed.

### 2. Re-emitted Events

The same event_id may appear multiple times with different ingestion timestamps.

Business Rule:

> The record with the latest ingest_time_utc is considered the latest version and is retained.

Implementation:

- Remove exact duplicates using dropDuplicates()
- Use row_number() window function partitioned by event_id
- Order by ingest_time_utc descending
- Keep only row_number = 1

In [0]:
# Remove exact duplicates

clean_df = clean_df.dropDuplicates()


from pyspark.sql.window import Window

# For each event_id, keep the record with the latest ingest_time_utc
window_spec = Window.partitionBy(
    "event_id"
).orderBy(
    F.col("ingest_time_utc").desc()
)


dedup_df = clean_df.withColumn(
    "row_number",
    F.row_number().over(window_spec)
)


final_events_df = dedup_df.filter(
    F.col("row_number") == 1
).drop("row_number")



## Silver Output

The cleaned event data is stored as:

`events_silver`

The Silver table contains:

- Standardized schema
- UTC normalized timestamps
- Valid records only (invalid timestamps and negative costs quarantined)
- No exact duplicates
- Latest version of each event_id

In [0]:
# Save Silver table
final_events_df.write.format("delta").mode("overwrite").saveAsTable("events_silver")

# Verify Silver table
events_silver_df = spark.read.table("events_silver")

events_silver_df.show(5)
events_silver_df.printSchema()

print("Silver record count:", events_silver_df.count())




+------------+----------+--------------------+-------------------+-------------+-----------+--------------+-------+-------+----------+--------+-----------+-------------------+
|    event_id|event_type|          event_time|        ingest_time|advertiser_id|campaign_id|     placement| device|    geo|media_cost|currency|viewability|    ingest_time_utc|
+------------+----------+--------------------+-------------------+-------------+-----------+--------------+-------+-------+----------+--------+-----------+-------------------+
|evt_1000d6e9|impression|2026-06-01T11:16:28Z|2026-06-02 18:51:55|     adv_1272| cmp_1272_3|banner_300x250|desktop|unknown|    0.0157|     USD|       NULL|2026-06-02 13:21:55|
|evt_10013404|impression|2026-06-04 02:01:...|2026-06-04 22:22:10|     adv_1328| cmp_1328_2|   native_feed| tablet|     AU|    0.0172|     USD|       0.49|2026-06-04 16:52:10|
|evt_1001a835|impression|2026-06-02T23:28:52Z|2026-06-02 13:56:45|     adv_1293| cmp_1293_1|   native_feed| mobile|     

#### Currency Conversion

Spend is reported in USD.

A fixed exchange rate is used:

1 USD = 83 INR

Only records with currency = 'INR' are converted to USD.
Records already in USD are retained without conversion.

In [0]:
from pyspark.sql import functions as F

# Read Silver events and advertiser dimension
events_silver_df = spark.read.table("events_silver")
advertisers_bronze_df = spark.read.table("advertisers_bronze")

# Join advertiser dimension
gold_df = events_silver_df.join(
    advertisers_bronze_df,
    on="advertiser_id",
    how="left"
)

# Handle missing advertisers
gold_df = gold_df.withColumn(
    "advertiser_name",
    F.coalesce(F.col("advertiser_name"), F.lit("UNKNOWN"))
)

# Convert only INR to USD (1 USD = 83 INR)
gold_df = gold_df.withColumn(
    "spend_usd",
    F.when(
        F.col("currency") == "INR",
        F.round(F.col("media_cost") / 83.0, 2)
    ).otherwise(F.col("media_cost"))
)

# Create event date
gold_df = gold_df.withColumn(
    "event_date",
    F.to_date(
        F.expr("try_to_timestamp(event_time)")
    )
)

# Calculate daily metrics per advertiser
gold_daily_df = gold_df.groupBy(
    "event_date",
    "advertiser_id",
    "advertiser_name"
).agg(
    F.sum("spend_usd").alias("daily_spend_usd"),
    F.count("event_id").alias("event_count")
)

# Save Gold table
gold_daily_df.write.format("delta").mode("overwrite").saveAsTable("gold_daily_advertiser_metrics")

# Verify output
gold_daily_df.show()
print("Gold record count:", gold_daily_df.count())

+----------+-------------+---------------+------------------+-----------+
|event_date|advertiser_id|advertiser_name|   daily_spend_usd|event_count|
+----------+-------------+---------------+------------------+-----------+
|2026-06-03|     adv_1328|     NovaRetail|           48.4697|        802|
|2026-06-04|     adv_1335|      KiteMedia| 29.18720000000004|        574|
|2026-06-06|     adv_1293|    NorthRetail| 40.85330000000001|        635|
|2026-06-05|     adv_1342|     NorthTrips|           31.0799|        545|
|2026-06-02|      adv_985|        UNKNOWN|1.7055999999999998|         57|
|2026-06-04|      adv_953|        UNKNOWN|            1.8018|         44|
|2026-06-04|     adv_1293|    NorthRetail| 35.84690000000003|        577|
|2026-06-02|     adv_1342|     NorthTrips|31.035799999999984|        641|
|2026-06-06|     adv_1328|     NovaRetail|28.872899999999976|        610|
|2026-06-07|     adv_1272|      BlueWorks| 30.24940000000002|        501|
|2026-06-04|     adv_1307|     AltaHea

# Step 4: Gold Layer

The Gold output is validated using the following checks:

- Schema verification
- Sample record review
- Duplicate grain validation
- Negative metric validation
- Missing advertiser handling validation

In [0]:
# Validate Gold table schema
gold_daily_df.printSchema()

# Display sample records
gold_daily_df.show(10, False)

print("Gold record count:", gold_daily_df.count())

# Verify there is only one record per advertiser per day
gold_daily_df.groupBy(
    "event_date",
    "advertiser_id"
).count().filter(
    F.col("count") > 1
).show()

# Verify there are no negative daily spend values
gold_daily_df.filter(
    F.col("daily_spend_usd") < 0
).show()

# Verify handling of advertisers missing from the dimension
gold_daily_df.filter(
    F.col("advertiser_name") == "UNKNOWN"
).show()

root
 |-- event_date: date (nullable = true)
 |-- advertiser_id: string (nullable = true)
 |-- advertiser_name: string (nullable = false)
 |-- daily_spend_usd: double (nullable = true)
 |-- event_count: long (nullable = false)

+----------+-------------+---------------+------------------+-----------+
|event_date|advertiser_id|advertiser_name|daily_spend_usd   |event_count|
+----------+-------------+---------------+------------------+-----------+
|2026-06-03|adv_1328     |NovaRetail     |48.4697           |802        |
|2026-06-04|adv_1335     |KiteMedia      |29.18720000000004 |574        |
|2026-06-06|adv_1293     |NorthRetail    |40.85330000000001 |635        |
|2026-06-05|adv_1342     |NorthTrips     |31.0799           |545        |
|2026-06-02|adv_985      |UNKNOWN        |1.7055999999999998|57         |
|2026-06-04|adv_953      |UNKNOWN        |1.8018            |44         |
|2026-06-04|adv_1293     |NorthRetail    |35.84690000000003 |577        |
|2026-06-02|adv_1342     |NorthT

# Pipeline Summary

## Bronze
- Loaded raw event and advertiser data into Delta tables.
- Preserved the original source data without transformations.

## Silver
- Handled schema differences between event versions.
- Standardized the schema across both source files.
- Normalized timestamps to UTC.
- Quarantined malformed timestamps and negative media costs.
- Removed exact duplicate records.
- Retained the latest version of duplicate event_ids using ingest_time_utc.

## Gold
- Joined event data with the advertiser dimension using a LEFT JOIN.
- Preserved events with missing advertisers by assigning advertiser_name as "UNKNOWN".
- Converted INR spend to USD using a fixed exchange rate (1 USD = 83 INR).
- Generated daily advertiser spend and event count metrics.

## Output Tables

**Bronze**
- advertisers_bronze
- events_day1_bronze
- events_day2_bronze

**Silver**
- events_silver

**Quarantine**
- events_quarantine

**Gold**
- gold_daily_advertiser_metrics

# Operational Notes

## Partitioning Strategy

- Bronze: Keep unpartitioned because it stores raw source data and is primarily used for ingestion.
- Silver: Partition by event_date as the dataset grows to improve query performance for date-based filtering.
- Gold: Partition by event_date since reporting queries are typically run by day.

## OPTIMIZE and Z-ORDER

- Run OPTIMIZE periodically to compact many small Delta files into fewer larger files.
- Apply Z-ORDER on advertiser_id and event_date to improve data skipping and speed up reporting queries.

## Incremental Processing

When a new day-8 file arrives:

- Ingest the new file into the Bronze layer.
- Process only the newly arrived records into Silver.
- Reapply data quality validation, timestamp normalization, and deduplication.
- If duplicate event_ids are received (late-arriving events), retain the latest record based on ingest_time_utc.
- Update affected Gold aggregates incrementally instead of rebuilding the entire pipeline.

# Step 5: SQL Analytics

The following SQL queries answer the analytical questions using the Silver and Gold tables created in this notebook.

In [0]:
%sql

SELECT
    event_date,
    advertiser_id,
    advertiser_name,
    daily_spend_usd,

    AVG(daily_spend_usd) OVER (
        PARTITION BY advertiser_id
        ORDER BY event_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7day_avg,

    ROUND(
        (
            (daily_spend_usd -
             LAG(daily_spend_usd) OVER (
                 PARTITION BY advertiser_id
                 ORDER BY event_date
             ))
            /
            LAG(daily_spend_usd) OVER (
                PARTITION BY advertiser_id
                ORDER BY event_date
            )
        ) * 100,
        2
    ) AS pct_change

FROM gold_daily_advertiser_metrics
ORDER BY advertiser_id, event_date;

event_date,advertiser_id,advertiser_name,daily_spend_usd,rolling_7day_avg,pct_change
2026-06-01,adv_1265,PrimeLabs,37.37289999999998,37.37289999999998,null
2026-06-02,adv_1265,PrimeLabs,30.958399999999994,34.165649999999985,-17.16
2026-06-03,adv_1265,PrimeLabs,47.42770000000001,38.58633333333333,53.2
2026-06-04,adv_1265,PrimeLabs,26.297899999999967,35.51422499999999,-44.55
2026-06-05,adv_1265,PrimeLabs,35.1165,35.43467999999999,33.53
2026-06-06,adv_1265,PrimeLabs,41.16730000000001,36.390116666666664,17.23
2026-06-07,adv_1265,PrimeLabs,27.67970000000001,35.14577142857143,-32.76
2026-06-01,adv_1272,BlueWorks,43.783500000000004,43.783500000000004,null
2026-06-02,adv_1272,BlueWorks,37.08220000000002,40.432850000000016,-15.31
2026-06-03,adv_1272,BlueWorks,51.84010000000001,44.23526666666668,39.8


In [0]:
%sql

WITH active_days AS (

    SELECT DISTINCT
        advertiser_id,
        event_date
    FROM gold_daily_advertiser_metrics

),

numbered AS (

    SELECT
        advertiser_id,
        event_date,
        ROW_NUMBER() OVER (
            PARTITION BY advertiser_id
            ORDER BY event_date
        ) AS rn
    FROM active_days

),

grouped AS (

    SELECT
        advertiser_id,
        event_date,
        DATE_SUB(event_date, rn) AS grp
    FROM numbered

)

SELECT
    advertiser_id,
    MIN(event_date) AS streak_start,
    MAX(event_date) AS streak_end,
    COUNT(*) AS longest_streak
FROM grouped
GROUP BY advertiser_id, grp
ORDER BY longest_streak DESC;

advertiser_id,streak_start,streak_end,longest_streak
adv_1328,2026-06-01,2026-06-07,7
adv_953,2026-06-01,2026-06-07,7
adv_1307,2026-06-01,2026-06-07,7
adv_1321,2026-06-01,2026-06-07,7
adv_1335,2026-06-01,2026-06-07,7
adv_1342,2026-06-01,2026-06-07,7
adv_1265,2026-06-01,2026-06-07,7
adv_1286,2026-06-01,2026-06-07,7
adv_1279,2026-06-01,2026-06-07,7
adv_1314,2026-06-01,2026-06-07,7


## SQL Question 3 - MERGE for Late-arriving Corrections

The following MERGE statement demonstrates how late-arriving correction records would be upserted into the Silver table.

The source table `corrections_batch` is assumed to contain incoming correction records.

In [0]:
%sql

MERGE INTO events_silver AS target

USING corrections_batch AS source

ON target.event_id = source.event_id

WHEN MATCHED
AND source.ingest_time_utc > target.ingest_time_utc

THEN UPDATE SET
    target.event_type = source.event_type,
    target.event_time = source.event_time,
    target.ingest_time = source.ingest_time,
    target.ingest_time_utc = source.ingest_time_utc,
    target.advertiser_id = source.advertiser_id,
    target.campaign_id = source.campaign_id,
    target.placement = source.placement,
    target.device = source.device,
    target.geo = source.geo,
    target.media_cost = source.media_cost,
    target.currency = source.currency,
    target.viewability = source.viewability

WHEN NOT MATCHED

THEN INSERT (
    event_id,
    event_type,
    event_time,
    ingest_time,
    ingest_time_utc,
    advertiser_id,
    campaign_id,
    placement,
    device,
    geo,
    media_cost,
    currency,
    viewability
)

VALUES (
    source.event_id,
    source.event_type,
    source.event_time,
    source.ingest_time,
    source.ingest_time_utc,
    source.advertiser_id,
    source.campaign_id,
    source.placement,
    source.device,
    source.geo,
    source.media_cost,
    source.currency,
    source.viewability
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7617747459018045>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '\nMERGE INTO events_silver AS target\n\nUSING corrections_batch AS source\n\nON target.event_id = source.event_id\n\nWHEN MATCHED\nAND source.ingest_time_utc > target.ingest_time_utc\n\nTHEN UPDATE SET\n    target.event_type = source.event_type,\n    target.event_time = source.event_time,\n    target.ingest_time = source.ingest_time,\n    target.ingest_time_utc = source.ingest_time_utc,\n    target.advertiser_id = source.advertiser_id,\n    target.campaign_id = source.campaign_id,\n    target.placement = source.placement,\n    target.device = source.device,\n    target.geo = source.geo,\n    target.media_cost = source.media_cost,\n    target.currency = source.currency,\n    target.viewability = source.viewability\n\nWHEN NOT MATCHED\n\nTHEN INSERT (

# Step 6: Data Quality Framework

### Objective

Create a reusable utility to validate data quality using configurable rules.

Supported checks:

- Null-rate threshold
- Uniqueness
- Freshness (maximum allowed age of event timestamps)

The function returns a structured pass/fail report that can be reused across different datasets.

In [0]:
#Python (Entire Function)

from pyspark.sql import functions as F
from datetime import datetime, timezone

def run_dq_checks(df, rules):
    """
    Runs configurable data quality checks on a PySpark DataFrame.

    Supported rules:
    - null_threshold
    - unique
    - freshness
    """

    report = []

    total_rows = df.count()

    # -------------------------
    # Null-rate checks
    # -------------------------
    for column, threshold in rules.get("null_threshold", {}).items():

        null_count = df.filter(F.col(column).isNull()).count()

        null_rate = (
            null_count / total_rows
            if total_rows > 0
            else 0
        )

        report.append({
            "check": f"Null Rate ({column})",
            "status": "PASS" if null_rate <= threshold else "FAIL",
            "observed": round(null_rate, 4),
            "threshold": threshold
        })

    # -------------------------
    # Uniqueness checks
    # -------------------------
    for column in rules.get("unique", []):

        duplicate_count = (
            df.groupBy(column)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

        report.append({
            "check": f"Uniqueness ({column})",
            "status": "PASS" if duplicate_count == 0 else "FAIL",
            "observed": duplicate_count,
            "threshold": 0
        })

    # -------------------------
    # Freshness check
    # -------------------------
    freshness = rules.get("freshness")

    if freshness:

        latest = df.select(
            F.max(freshness["column"])
        ).first()[0]

        if latest is None:
            age = None
            status = "FAIL"

        else:
            age = (
                datetime.now(timezone.utc)
                - latest.replace(tzinfo=timezone.utc)
            ).days

            status = (
                "PASS"
                if age <= freshness["max_age_days"]
                else "FAIL"
            )

        report.append({
            "check": f"Freshness ({freshness['column']})",
            "status": status,
            "observed": age,
            "threshold": freshness["max_age_days"]
        })

    return report

In [0]:
# example usage
dq_rules = {

    "null_threshold": {
        "geo": 0.05,
        "device": 0.02
    },

    "unique": [
        "event_id"
    ],

    "freshness": {
        "column": "ingest_time_utc",
        "max_age_days": 30
    }

}

dq_report = run_dq_checks(
    events_silver_df,
    dq_rules
)

for result in dq_report:
    print(result)

## Unit Tests

The following unit tests validate important edge cases for the reusable data quality utility.

In [0]:
# ------------------------------------
# Test 1 : Duplicate event_id
# Expected : FAIL
# ------------------------------------

test_df = spark.createDataFrame(
    [
        ("e1",),
        ("e1",)
    ],
    ["event_id"]
)

result = run_dq_checks(
    test_df,
    {
        "unique": ["event_id"]
    }
)

assert result[0]["status"] == "FAIL"

print("✓ Test 1 Passed")


# ------------------------------------
# Test 2 : Empty DataFrame
# Expected : PASS
# ------------------------------------

empty_df = spark.createDataFrame(
    [],
    "id INT"
)

result = run_dq_checks(
    empty_df,
    {
        "null_threshold": {
            "id": 0.10
        }
    }
)

assert result[0]["status"] == "PASS"

print("✓ Test 2 Passed")


# ------------------------------------
# Test 3 : Null rate exceeds threshold
# Expected : FAIL
# ------------------------------------

test_df = spark.createDataFrame(
    [
        (1,),
        (None,),
        (None,)
    ],
    ["value"]
)

result = run_dq_checks(
    test_df,
    {
        "null_threshold": {
            "value": 0.50
        }
    }
)

assert result[0]["status"] == "FAIL"

print("✓ Test 3 Passed")

print("\n🎉 All unit tests passed successfully.")

✓ Test 1 Passed
✓ Test 2 Passed
✓ Test 3 Passed

🎉 All unit tests passed successfully.


# Conclusion

This notebook implements an end-to-end data pipeline using the Medallion Architecture approach.

## Key Outcomes

- Ingested raw vendor files into Bronze Delta tables while preserving source data.
- Applied Silver layer transformations including schema alignment, data validation, quarantine handling, timestamp standardization and deduplication.
- Created Gold layer business metrics by combining event data with advertiser information and generating daily spend and event count aggregations.
- Added analytical SQL queries for reporting use cases.
- Implemented reusable data quality checks with configurable rules and unit tests.

## Production Considerations

For a production implementation:
- Use incremental ingestion instead of full reloads.
- Use Delta MERGE for handling late-arriving corrections.
- Add monitoring and alerting for pipeline failures and data quality issues.
- Apply appropriate partitioning and optimization strategies based on data volume.

The final pipeline provides reliable, validated and analytics-ready data for downstream reporting.

# Written Assessment

## README: Approach, Assumptions and Trade-offs

### Approach

I approached this task by following a simple medallion architecture pattern.

First, I ingested the raw CSV files into the Bronze layer without making changes to the source data. This helps preserve the original data for auditing and future debugging.

In the Silver layer, I focused on preparing the data for analysis. This included handling schema differences between files, standardizing column names and data types, validating required fields, removing duplicate records, and identifying invalid records.

Finally, in the Gold layer, I created reporting-ready data by applying aggregations required for analysis.

### Assumptions

- I considered event_id as the unique identifier for an event.
- Records with missing important fields such as event_id or spend were treated as invalid.
- Duplicate records were removed to avoid incorrect reporting.
- Invalid records were separated instead of deleting them permanently, so they can be reviewed later.

### Trade-offs

- Additional validation improves data quality but increases processing time.
- Keeping invalid records separately requires extra storage but helps with troubleshooting.
- Delta Lake provides reliability and data versioning, but it needs more storage compared to plain files.

---

## Client Communication

Hi Team,

We identified an issue with the vendor file received tonight. The file appears to be corrupted, and we are currently unable to complete the data processing successfully.

Due to this issue, the 9 AM dashboard refresh may be delayed. We are working on identifying the root cause and coordinating for a corrected file.

We will provide an update as soon as we have more information and share the revised timeline.

Thanks,
Anitha

---

## Code Review

The main issues I identified in the code are:

1. The schema is not defined while reading the CSV file. Relying on automatic schema inference can create data type issues and may be slower for large files.

2. `dropDuplicates()` is applied without specifying the key column. For large datasets, this can be expensive. Deduplication should usually be based on a business key like event_id.

3. The null check is incorrect. In Spark, null values should be handled using functions like `isNull()` or `isNotNull()`.

4. `collect()` should be avoided because it brings all data to the driver. With large datasets, this can cause memory issues.

5. Processing rows one by one using a loop is not suitable for Spark. Spark works best with distributed DataFrame transformations.

6. Filtering inside the loop creates multiple Spark operations and will reduce performance.

7. Caching the dataframe without knowing whether it will be reused may unnecessarily consume memory.

---

## Scale-up Considerations

If this feed becomes 10 times larger and arrives hourly, I would first focus on improving the ingestion and processing approach.

The changes I would consider are:

- Process only new incoming files instead of reprocessing all historical data every hour.
- Use Delta Lake features for better reliability and incremental processing.
- Add partitioning based on date columns to reduce data scanning.
- Optimize duplicate handling instead of performing expensive full-data comparisons.
- Add automated data quality checks for schema changes, missing values and invalid records.
- Monitor pipeline execution time, failures and record counts.
- Optimize storage by managing small files generated from frequent loads.